# Qwen 结构与推理探索：从加载到 KV Cache

这份 Notebook 将 MiniGPT 中已经学习过的模块，与真实预训练模型的对象、参数和推理数据流对应起来。讲解严格按现有代码顺序展开；沿用“模块作用 → 变量含义 → shape 变化 → 已有结果解读”的组织方式。

学习路径：**加载与结构 → generate 接口 → GQA 投影 → 第一层 Attention → 最终隐藏状态与 lm_head → 手动生成 → KV Cache → 性能测量**。

统一记号：`B` 为 Batch，`T` 为当前输入长度，`D` 为隐藏维度，`V` 为词表输出维度；`Hq` 为 Query 头数，`Hkv` 为 Key/Value 头数，`Dh` 为单头维度。本例保存的值为 `B=1, T=7, D=896, V=151936, Hq=14, Hkv=2, Dh=64`。

文中的实际数值均来自原有输出；未打印的中间形状依据代码推导。本次没有执行 Notebook，也没有修改原有代码、输出或执行状态。

## 1. 加载 Tokenizer 和模型

### 导入计算框架与自动加载接口

下面的 Cell 只导入依赖：`torch` 提供张量和推理上下文，`AutoTokenizer` 加载配套文本编码器，`AutoModelForCausalLM` 加载带词表预测头的因果语言模型。实际加载资源发生在后面两个 Cell。

“Auto”表示根据配置选择具体实现类，不表示自动训练；“CausalLM”表示利用截至当前位置的可见前缀预测下一个 Token。

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

### 1.1 模型名称、Tokenizer 与文本编码

`model_name` 指向 `Qwen/Qwen2.5-0.5B` 基础模型。Tokenizer 与模型应配套，保证整数 ID 与模型中的 Embedding、词表输出含义一致。

`AutoTokenizer.from_pretrained(model_name)` 读取分词器资源；`tokenizer(text, return_tensors="pt")` 把文本编码为 PyTorch 张量，并放入 `inputs` 这个映射对象。

已有输出为 `[9707, 11, 358, 1079, 264, 5458, 13]`，即 7 个 Token。因此 `inputs["input_ids"]` 是 `[1, 7]`；`inputs["attention_mask"]` 同样是 `[1, 7]` 且全为 1，说明本例没有 padding。

7 来自当前 Tokenizer 的编码结果，不是英文单词数量。当前代码直接使用普通文本，没有调用对话模板；后续示例都围绕这一输入进行。

In [3]:
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "Hello, I am a student."
inputs = tokenizer(text, return_tensors="pt")

print("input text:")
print(text)

print("\ninput_ids:")
print(inputs["input_ids"])

print("\nattention_mask:")
print(inputs["attention_mask"])

input text:
Hello, I am a student.

input_ids:
tensor([[9707,   11,  358, 1079,  264, 5458,   13]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1]])


### 1.2 加载预训练权重

`AutoModelForCausalLM.from_pretrained(model_name)` 根据配置创建模型对象，从本地缓存或模型仓库读取预训练权重。这里加载的是已经训练好的参数，不进行训练或梯度更新。

已有输出 `Model loaded successfully.` 表示保存时完成了该加载步骤；它没有展示预测，也没有测量加载时间。下一节通过 `print(model)` 阅读实际结构。

In [4]:
model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded successfully.")

Model loaded successfully.


## 2. 理解 Qwen 的整体结构

下面打印的是模型对象树。先区分外层语言模型封装与内层主体：

```text
Qwen2ForCausalLM
├── model: Qwen2Model
│   ├── embed_tokens: Embedding(151936, 896)
│   ├── layers: 24 个 Qwen2DecoderLayer
│   ├── norm: Qwen2RMSNorm
│   └── rotary_emb: Qwen2RotaryEmbedding
└── lm_head: Linear(896, 151936)
```

`embed_tokens` 将 ID 映射为向量，`layers` 逐层更新上下文表示，主体末尾的 `norm` 处理最终隐藏状态，外层 `lm_head` 将每个位置映射为词表分数。

**模块打印顺序不等于 forward 的执行顺序。** 例如 `rotary_emb` 虽然列在 `norm` 后面，却用于生成 Attention 中 Q/K 的旋转位置信息，不是在最后给隐藏状态加位置向量。

已有输出 `(0-23): 24 x Qwen2DecoderLayer` 表示 24 个结构相同的层实例，各自具有对应层的参数，不是一个层的计算结果重复使用 24 次。

In [5]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

### 2.1 核对 Decoder 层数

`model.model.layers` 是主体的层列表，`len(...)` 统计其中的层实例数量，不执行模型前向。已有输出为 **24**，与前面打印的索引 `0–23` 一致。

`model.model` 的第二个 `model` 是对象的属性名，指向 `Qwen2Model` 主体，不表示两个完整模型串联。

In [6]:
num_layers = len(model.model.layers)

print("Number of decoder layers:")
print(num_layers)

Number of decoder layers:
24


### 2.2 阅读第一层 Qwen2DecoderLayer

`first_layer = model.model.layers[0]` 取得第 0 层；下面的 Cell 只打印它，不执行这一层。

| 实际模块名 | 职责与维度 |
|---|---|
| `input_layernorm` | Attention 前的 RMSNorm，输入输出均为 `[B, T, 896]` |
| `self_attn` | 因果自注意力，含 `q_proj`、`k_proj`、`v_proj` 和 `o_proj`，负责可见 Token 间的交互 |
| `post_attention_layernorm` | Attention 残差相加之后、MLP 之前的 RMSNorm，shape 不变 |
| `mlp` | 逐位置门控特征变换：`gate_proj` 和 `up_proj` 都是 `896 → 4864`，`down_proj` 是 `4864 → 896`，激活为 SiLU |

名称含 `layernorm`，实际打印的类型却是 `Qwen2RMSNorm`：依据均方根缩放特征，不等同于 MiniGPT 使用的 `nn.LayerNorm`。

层内的计算关系是：**输入 → input_layernorm → self_attn → 与原输入残差相加 → post_attention_layernorm → mlp → 再次残差相加**。这是 Pre-Norm，与 MiniGPT 的 Post-LayerNorm 顺序不同。MLP 的两条升维分支通过 SiLU 门控和逐元素乘法结合，再降维，不是原 MiniGPT 的 Linear–ReLU–Linear。

残差相加是 forward 中的张量运算，不一定以独立模块出现在对象树里。后面的手动 Attention 示例只展开其中一部分。

In [7]:
first_layer = model.model.layers[0]

print(first_layer)

Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=896, out_features=896, bias=True)
    (k_proj): Linear(in_features=896, out_features=128, bias=True)
    (v_proj): Linear(in_features=896, out_features=128, bias=True)
    (o_proj): Linear(in_features=896, out_features=896, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)


## 3. 使用 model.generate() 生成文本

这个 Cell 把一次预测扩展为连续续写：

- `model.eval()` 设置评估模式；`torch.inference_mode()` 关闭梯度记录并减少推理相关开销，二者作用不同。
- `next(model.parameters()).device` 读取参数设备；`inputs.to(...)` 把输入张量放到相同设备，并不主动迁移模型。
- `**model_inputs` 将 `input_ids`、`attention_mask` 等字段展开为关键字参数。
- `max_new_tokens=12` 是新增 Token 的上限，若遇到生效的停止条件，可以提前结束；本次保存结果实际新增了 12 个。
- `do_sample=False` 关闭随机采样；本例后续手动循环采用贪心选择。当前 Cell 没有另行指定 beam 参数。
- `pad_token_id=tokenizer.eos_token_id` 设置生成接口使用的 padding ID，不表示主动给输入追加 EOS。

Decoder-only 模型的 `generated_ids` 包含 **原始 Prompt + 新增 Token**。`prompt_len` 保存原输入长度，用 `generated_ids[:, prompt_len:]` 才切出新增部分。`skip_special_tokens=True` 影响解码显示，不改变返回张量的长度。

已有输出：输入 `[1, 7]` → 完整生成序列 `[1, 19]`，新增部分为 `[1, 12]`；新增文本为 ` I am studying for my final exam. I have a question`。这段结果来自已保存输出，没有重新生成。

In [8]:
model.eval()
model_inputs = inputs.to(next(model.parameters()).device)

with torch.inference_mode():
    generated_ids = model.generate(**model_inputs, max_new_tokens=12, do_sample=False, pad_token_id=tokenizer.eos_token_id)

prompt_len = model_inputs["input_ids"].shape[1]
new_token_ids = generated_ids[:, prompt_len:]
full_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
new_text = tokenizer.decode(new_token_ids[0], skip_special_tokens=True)

print("原始输入：", text)
print("输入形状：", tuple(model_inputs["input_ids"].shape))
print("生成后形状：", tuple(generated_ids.shape))
print("新生成 Token 数量：", new_token_ids.shape[1])

print("\n完整生成文本：")
print(full_text)

print("\n仅新生成的内容：")
print(new_text)

原始输入： Hello, I am a student.
输入形状： (1, 7)
生成后形状： (1, 19)
新生成 Token 数量： 12

完整生成文本：
Hello, I am a student. I am studying for my final exam. I have a question

仅新生成的内容：
 I am studying for my final exam. I have a question


## 4. 理解 GQA 和投影层维度

当前模型使用 Grouped-Query Attention（GQA）：Query 头较多，一组 Query 头共享同一个 Key/Value 头。下面读取配置，并检查真实投影层的输入输出维度。

| 变量或维度 | 保存的值 | 关系 |
|---|---:|---|
| `hidden_size` | 896 | 每个位置输入特征的维度 |
| `num_attention_heads` | 14 | Query 头数 |
| `num_key_value_heads` | 2 | Key、Value 各自的头数 |
| `head_dim` | 64 | `896 // 14` |
| Q 投影输出维度 | 896 | `14 × 64` |
| K、V 各自的投影输出维度 | 128 | `2 × 64` |

Q/K/V 都读取同一个 896 维输入，因此 `in_features` 相同；输出维度按各自的头数打包。因此 `q_proj` 是 `896 → 896`，`k_proj`、`v_proj` 都是 `896 → 128`。

对 `[1, 7, 896]` 的输入，Q 投影得到 `[1, 7, 896]`，K、V 各得到 `[1, 7, 128]`。**128 不是模型隐藏维度变成了 128，而是仅有 2 个 KV 头，每头 64 维。**

`14 / 2 = 7`，所以每个 KV 头供一组 7 个 Query 头使用。下一节通过 `repeat_interleave` 显式展开共享关系，不会新增 14 套 KV 投影参数。后面的 KV Cache 也可以保存较少的 KV 头，而非为每个 Query 头独立存一份。

In [9]:
first_layer = model.model.layers[0]
attn = first_layer.self_attn

hidden_size = model.config.hidden_size
num_attention_heads = model.config.num_attention_heads
num_key_value_heads = model.config.num_key_value_heads
head_dim = hidden_size // num_attention_heads

print("隐藏维度 hidden_size:", hidden_size)
print("Query 头数量:", num_attention_heads)
print("Key/Value 头数量:", num_key_value_heads)
print("每个注意力头的维度:", head_dim)

print("\nQ 投影层:", attn.q_proj.in_features, "->", attn.q_proj.out_features)
print("K 投影层:", attn.k_proj.in_features, "->", attn.k_proj.out_features)
print("V 投影层:", attn.v_proj.in_features, "->", attn.v_proj.out_features)

隐藏维度 hidden_size: 896
Query 头数量: 14
Key/Value 头数量: 2
每个注意力头的维度: 64

Q 投影层: 896 -> 896
K 投影层: 896 -> 128
V 投影层: 896 -> 128


## 5. 手动展开第一层 Attention

**下面只展开第 0 层 Attention 的核心计算，不是完整 Decoder Layer，也不是完整 Qwen 前向。它没有加入该层完整残差、后续 MLP、其余 23 层，以及主体最终 RMSNorm 和 lm_head。**

本节重新从 `inputs["input_ids"]` 取原输入，不使用前面 `generate()` 得到的续写序列。`device` 使输入与参数位于同一设备，计算位于 `torch.inference_mode()` 内。

### 5.1 Embedding 与 input_layernorm

`batch_size, seq_len = input_ids.shape` 得到 `1, 7`。Embedding 把 `[1, 7]` 映射为 `[1, 7, 896]`；`first_layer.input_layernorm` 对每个位置做 RMSNorm，shape 不变。

`hidden_states` 随后被归一化结果覆盖，因此末尾打印的是**归一化后的输入**，不是原始 Embedding。这里尚未加入层内残差。

### 5.2 Q/K/V 的线性投影与分头

| 阶段 | Q 的 shape | K、V 各自的 shape |
|---|---|---|
| 投影后 | `[1, 7, 896]` | `[1, 7, 128]` |
| `view(batch_size, seq_len, heads, head_dim)` | `[1, 7, 14, 64]` | `[1, 7, 2, 64]` |
| `transpose(1, 2)` | `[1, 14, 7, 64]` | `[1, 2, 7, 64]` |

`view()` 把打包的特征轴拆成“头数 × 单头维度”，元素总数不变；`transpose()` 交换头轴和序列轴，使各头能够独立做矩阵乘法。未打印的投影中间形状依据上述代码推导。

### 5.3 RoPE：为 Q/K 加入位置信息

`position_ids = torch.arange(seq_len, device=device).unsqueeze(0)` 生成 `[1, 7]` 的位置编号 `0…6`。`model.model.rotary_emb(hidden_states, position_ids)` 返回相应 `cos`、`sin`；按此调用推导，二者为 `[1, 7, 64]`，`unsqueeze(1)` 后为 `[1, 1, 7, 64]`，可广播到全部注意力头。

本 Cell 的 `rotate_half(x)` 将最后 64 维分为前后各 32 维，再组合为“负后半、前半”。原向量乘 cos，加上旋转向量乘 sin，构成随位置变化的旋转，使后续 Q/K 点积体现相对位置信息。

Q、K 的 shape 不变；V 不做这一旋转。这不是 MiniGPT 中给 Embedding 直接加位置向量的方式。

### 5.4 GQA：扩展共享的 K/V 头

`num_kv_groups = num_heads // num_kv_heads = 7`。`repeat_interleave(num_kv_groups, dim=1)` 将每个 KV 头连续重复 7 次：

`K/V: [1, 2, 7, 64] → [1, 14, 7, 64]`

前 7 个 Query 头使用第一组 KV，后 7 个使用第二组。这个示例显式扩展张量以便观察计算，不会创建新参数，也不能据此认为模型的历史缓存必须存 14 个 KV 头。

### 5.5 Attention Score、因果遮罩与 Softmax

`q @ k.transpose(-2, -1) / math.sqrt(head_dim)` 执行：

`[1, 14, 7, 64] @ [1, 14, 64, 7] → scores [1, 14, 7, 7]`

最后两轴分别表示 Query 位置和 Key 位置；除以 `sqrt(64)` 缩放点积。此时是匹配分数，尚不是概率权重。

`torch.triu(..., diagonal=1)` 生成 `[7, 7]` 的布尔上三角遮罩，**True 表示未来位置，应被屏蔽**。`masked_fill` 将这些分数设为负无穷，随后 `softmax(scores, dim=-1)` 沿 Key 轴归一化，得到 `weights [1, 14, 7, 7]`。未来位置权重为 0，每个 Query 对可见位置的权重和为 1。

这里的 True 表示屏蔽，与 MiniGPT 中“下三角 1 表示可见”的写法判定方向不同，但保留的可见区域相同。本 Cell 只有 causal mask，没有合并 padding mask；当前输入没有 padding，所以讲解限定于这一示例。

### 5.6 Weighted Sum、contiguous 与 o_proj

`weights @ v` 将 `[1, 14, 7, 7]` 与 `[1, 14, 7, 64]` 相乘，输出 `[1, 14, 7, 64]`，即每个头对可见 Value 的加权汇总。

随后 `transpose(1, 2)` 得到 `[1, 7, 14, 64]`。转置通常改变内存步长；`contiguous()` 让结果具有连续布局，便于后面的 `view(batch_size, seq_len, hidden_size)` 合头为 `[1, 7, 896]`。

`attn.o_proj` 再做 `896 → 896` 的线性投影，混合各头的信息，shape 不变。**o_proj 是 Attention 的输出投影，不是映射到 151936 个词表分数的 lm_head。**

已有打印依次确认：

- `hidden_states`：`[1, 7, 896]`。
- `q` 和扩展后的 `k`、`v`：`[1, 14, 7, 64]`。
- `scores`、`weights`：`[1, 14, 7, 7]`；打印时的 scores 已应用遮罩。
- `attn_output`：`[1, 7, 896]`。

这些输出展示了保存时的形状；当前代码没有将手动结果与模型内部 Attention 或完整 Decoder 的输出进行数值一致性比较。

In [10]:
import math
import torch

device = next(model.parameters()).device
first_layer = model.model.layers[0]
attn = first_layer.self_attn

hidden_size = model.config.hidden_size
num_heads = model.config.num_attention_heads
num_kv_heads = model.config.num_key_value_heads
head_dim = hidden_size // num_heads
num_kv_groups = num_heads // num_kv_heads

input_ids = inputs["input_ids"].to(device)
batch_size, seq_len = input_ids.shape

with torch.inference_mode():
    hidden_states = model.model.embed_tokens(input_ids)
    hidden_states = first_layer.input_layernorm(hidden_states)

    q = attn.q_proj(hidden_states)
    k = attn.k_proj(hidden_states)
    v = attn.v_proj(hidden_states)

    q = q.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
    k = k.view(batch_size, seq_len, num_kv_heads, head_dim).transpose(1, 2)
    v = v.view(batch_size, seq_len, num_kv_heads, head_dim).transpose(1, 2)

    position_ids = torch.arange(seq_len, device=device).unsqueeze(0)
    cos, sin = model.model.rotary_emb(hidden_states, position_ids)

    def rotate_half(x):
        x1 = x[..., :head_dim // 2]
        x2 = x[..., head_dim // 2:]
        return torch.cat((-x2, x1), dim=-1)

    q = q * cos.unsqueeze(1) + rotate_half(q) * sin.unsqueeze(1)
    k = k * cos.unsqueeze(1) + rotate_half(k) * sin.unsqueeze(1)
    k = k.repeat_interleave(num_kv_groups, dim=1)
    v = v.repeat_interleave(num_kv_groups, dim=1)

    scores = q @ k.transpose(-2, -1) / math.sqrt(head_dim)
    causal_mask = torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=device),
        diagonal=1
    )
    scores = scores.masked_fill(causal_mask, float("-inf"))

    weights = torch.softmax(scores, dim=-1)
    attn_output = weights @ v

    attn_output = attn_output.transpose(1, 2).contiguous()
    attn_output = attn_output.view(batch_size, seq_len, hidden_size)
    attn_output = attn.o_proj(attn_output)

print("hidden_states:", tuple(hidden_states.shape))
print("Q:", tuple(q.shape))
print("K、V（共享扩展后）:", tuple(k.shape), tuple(v.shape))
print("scores:", tuple(scores.shape))
print("weights:", tuple(weights.shape))
print("Attention 输出:", tuple(attn_output.shape))

hidden_states: (1, 7, 896)
Q: (1, 14, 7, 64)
K、V（共享扩展后）: (1, 14, 7, 64) (1, 14, 7, 64)
scores: (1, 14, 7, 7)
weights: (1, 14, 7, 7)
Attention 输出: (1, 7, 896)


## 6. 最终隐藏状态、lm_head 与 Logits

本节回到完整主体。`model.model(**model_inputs)` 从原始输入开始，执行 Embedding、全部 24 层 Decoder 和主体最终 RMSNorm，**没有接着使用上一节的 attn_output**。

- `model.model(...)` 调用 `Qwen2Model`，返回 `base_outputs`。
- `base_outputs.last_hidden_state` 是全部位置最终的上下文表示，已经过主体末尾归一化；不是“最后一个 Token”的别名，也不是初始 Embedding。
- `model.lm_head(final_hidden_states)` 将每个位置的隐藏向量映射到词表分数。
- 外层 `model(...)`，即 `Qwen2ForCausalLM`，把主体与预测头封装在同一次调用中，返回 `outputs.logits`。

| 变量或索引 | 本例 shape | 含义 |
|---|---|---|
| `model_inputs["input_ids"]` | `[1, 7]` | 原始 Token IDs |
| `final_hidden_states` | `[1, 7, 896]` | 每个位置的最终表示 |
| `logits` | `[1, 7, 151936]` | 未归一化词表分数 |
| `logits[0, -1, :]` | `[151936]` | 第 0 个样本、最后位置、全部词表项 |
| `next_token_probs` | `[151936]` | 沿词表轴 Softmax 后的概率分布 |
| `top_probs`、`top_ids` | 各 `[5]` | 最高的五项概率与对应 ID |

整数索引 `0` 移除了 Batch 轴，所以 `next_token_logits` 是一维；这与保留 Batch 的 `logits[:, -1, :]` 不同。

**Logits 不等于概率。** Softmax 才将分数转换为和为 1 的分布。`topk(..., k=5)` 用来观察候选，不是从 Top-5 随机采样；最终仍通过 `argmax(next_token_logits)` 贪心选择。`.item()` 将单元素张量转换为 Python 标量。

已有输出最高项为 ID `358`、文本 `' I'`，概率约 `0.5397`；其余四项包括 `' My'`、`' Please'`、`' What'`、`' In'`。这五项只是完整分布的一部分，因此它们的概率不必合计为 1；打印时还做了四位小数舍入。

In [11]:
model.eval()
device = next(model.parameters()).device
model_inputs = inputs.to(device)

with torch.inference_mode():
    base_outputs = model.model(**model_inputs)
    final_hidden_states = base_outputs.last_hidden_state
    logits = model.lm_head(final_hidden_states)
    next_token_logits = logits[0, -1, :]
    next_token_probs = torch.softmax(next_token_logits, dim=-1)
    top_probs, top_ids = torch.topk(next_token_probs, k=5)
    next_token_id = torch.argmax(next_token_logits).item()

print("输入形状：", tuple(model_inputs["input_ids"].shape))
print("最终隐藏状态形状：", tuple(final_hidden_states.shape))
print("Logits 形状：", tuple(logits.shape))
print("最后一个位置的分数形状：", tuple(next_token_logits.shape))

print("\n概率最高的 5 个候选 Token：")
for token_id, prob in zip(top_ids.tolist(), top_probs.tolist()):
    print("Token ID:", token_id,
        "| Token:", repr(tokenizer.decode([token_id])),
        "| 概率:", round(prob, 4))

print("\n贪心选择的下一个 Token：")
print("Token ID:", next_token_id)
print("Token 文本:", repr(tokenizer.decode([next_token_id])))

输入形状： (1, 7)
最终隐藏状态形状： (1, 7, 896)
Logits 形状： (1, 7, 151936)
最后一个位置的分数形状： (151936,)

概率最高的 5 个候选 Token：
Token ID: 358 | Token: ' I' | 概率: 0.5397
Token ID: 3017 | Token: ' My' | 概率: 0.1001
Token ID: 5209 | Token: ' Please' | 概率: 0.0281
Token ID: 3555 | Token: ' What' | 概率: 0.022
Token ID: 758 | Token: ' In' | 概率: 0.0216

贪心选择的下一个 Token：
Token ID: 358
Token 文本: ' I'


## 7. 手动实现自回归生成：每轮输入完整序列

下面把一次预测放进循环。`generated_ids`、`attention_mask` 从原输入克隆，`prompt_len=7` 保存切分边界；本节重新从原句开始生成，不接着前面 `model.generate()` 的结果。

每轮按以下顺序执行：

1. 用完整的 `generated_ids` 和对应 `attention_mask` 做前向。
2. 取 `outputs.logits[:, -1, :]`，得到 `[1, 151936]` 的下一 Token 分数。
3. `argmax(..., dim=-1, keepdim=True)` 选择 `next_token_id [1, 1]`；保留长度轴便于拼接。
4. `torch.cat(..., dim=1)` 将新 Token 拼到完整序列末尾。
5. `ones_like(next_token_id)` 产生 `[1, 1]` 的有效位置标记，再拼到 `attention_mask`。
6. 解码并判断是否等于 `tokenizer.eos_token_id`；如果是，就提前退出。

| 轮次 | 本轮输入 shape | Logits shape | 拼接后的序列与 mask |
|---|---|---|---|
| 第 1 轮 | `[1, 7]` | `[1, 7, 151936]` | `[1, 8]` |
| 第 2 轮 | `[1, 8]` | `[1, 8, 151936]` | `[1, 9]` |
| 第 5 轮 | `[1, 11]` | `[1, 11, 151936]` | `[1, 12]` |

表中的中间 shape 按代码推导。已有输出显示五步依次为 `' I'`、`' am'`、`' studying'`、`' for'`、`' my'`，没有遇到 EOS，共新增 5 个 Token；最终序列长度为 `7+5=12`。

**这里准确的描述是“没有复用上一轮的 KV Cache”，不是“完全关闭缓存”。** 调用没有显式设置 `use_cache=False`；单次调用是否建立并返回缓存取决于配置与库实现。但是代码没有把 `outputs.past_key_values` 传回下一轮，所以每次仍输入整个增长中的序列，重新处理历史位置。

`next_token_id.item()` 和单个 EOS 判断针对当前 `B=1` 示例，没有实现多样本各自结束的逻辑。最后使用 `generated_ids[:, prompt_len:]` 单独解码新增内容。

In [12]:
model.eval()
device = next(model.parameters()).device

model_inputs = inputs.to(device)
generated_ids = model_inputs["input_ids"].clone()
attention_mask = model_inputs["attention_mask"].clone()
prompt_len = generated_ids.shape[1]

with torch.inference_mode():
    for step in range(5):
        outputs = model(input_ids=generated_ids, attention_mask=attention_mask)
        next_token_logits = outputs.logits[:, -1, :]
        next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        generated_ids = torch.cat([generated_ids, next_token_id], dim=1)
        new_attention_mask = torch.ones_like(next_token_id)
        attention_mask = torch.cat([attention_mask, new_attention_mask], dim=1)

        token_id = next_token_id.item()
        token_text = tokenizer.decode([token_id])

        print(f"第 {step + 1} 步生成：{token_text!r} (ID: {token_id})")
        if token_id == tokenizer.eos_token_id:
            print("模型生成了结束标记，停止生成。")
            break

new_token_ids = generated_ids[:, prompt_len:]

print("\n完整文本：")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

print("\n仅新生成的内容：")
print(tokenizer.decode(new_token_ids[0], skip_special_tokens=True))

print("\n原始输入长度：", prompt_len)
print("实际新增 Token 数量：", new_token_ids.shape[1])

第 1 步生成：' I' (ID: 358)
第 2 步生成：' am' (ID: 1079)
第 3 步生成：' studying' (ID: 20956)
第 4 步生成：' for' (ID: 369)
第 5 步生成：' my' (ID: 847)

完整文本：
Hello, I am a student. I am studying for my

仅新生成的内容：
 I am studying for my

原始输入长度： 7
实际新增 Token 数量： 5


## 8. 使用 KV Cache 生成：复用历史 K/V

与上一节相比，关键变化是把 `outputs.past_key_values` 传回下一次前向，而不只是声明 `use_cache=True`。

### 8.1 首次处理完整 Prompt：Prefill

循环前，`model(input_ids=generated_ids, attention_mask=attention_mask, use_cache=True)` 处理 `[1, 7]` 的原始 Prompt，得到预测首个新 Token 的 Logits，并返回 `outputs.past_key_values`。

KV Cache 按层保存历史位置的 Key、Value，供后续注意力复用；它不是词表 Logits，也不是单独一份最终隐藏状态。后续无需为同一段历史重新做全部投影和层间前向。

### 8.2 后续只输入一个新 Token：Decode

循环先读取当前 `outputs.logits[:, -1, :]`，选择新 Token 并加入完整 `generated_ids`。若不是 EOS，才延长 mask 并调用：

`model(input_ids=next_token_id, attention_mask=attention_mask, past_key_values=outputs.past_key_values, use_cache=True)`

这次 `input_ids` 是 `[1, 1]`，历史信息由缓存提供；该新 Token 的 Query 仍需关注历史 K/V 和自身，因此 mask 覆盖的是**历史长度 + 当前输入长度**。库依据缓存推进当前位置；后续返回的 Logits 在本例中为 `[1, 1, 151936]`。

| 阶段 | 送入模型的 IDs | 之前缓存长度 | 本次 mask 长度 | 本次计算后缓存长度 |
|---|---|---:|---:|---:|
| Prompt Prefill | `[1, 7]` | 0 | 7 | 7 |
| 处理第 1 个新 Token | `[1, 1]` | 7 | 8 | 8 |
| 处理第 2 个新 Token | `[1, 1]` | 8 | 9 | 9 |

上表依据现有调用顺序推导；代码没有直接打印缓存对象或缓存 shape。Cache 的具体容器/API 随 Transformers 版本而异，当前写法通过 `outputs.past_key_values` 原样传递它。

### 8.3 KV Cache 与 GQA

按结构推导，常规逐层 K、V 缓存各有 `[B, 2, L, 64]` 的逻辑维度，`L` 是已处理长度；不是手动 Attention 为计算而扩展的 `[B, 14, L, 64]`。两个 KV 头供 14 个 Query 头共享，减少了要缓存的 KV 元素数；不代表总模型内存或推理时间也同比例下降。

Cache 减少历史表示的重复计算，但新 Query 仍要与可见历史 K/V 交互。已有输出同样得到五个新 Token，最终 `generated_ids` 为 `[1, 12]`；这一 Cell 没有进行与上一节的计时比较。

**留意实际循环顺序：** 第 5 次选出 Token 后，若它不是 EOS，代码仍执行一次缓存前向；这次预测不会再被下一轮读取。因此当前示例多做了一次末尾未消费预测的前向，不能直接当作最精简的五步生成计时实现。这里如实解释，原代码保持不变。

In [13]:
model.eval()
device = next(model.parameters()).device
model_inputs = inputs.to(device)
generated_ids = model_inputs["input_ids"].clone()
attention_mask = model_inputs["attention_mask"].clone()

with torch.inference_mode():
    outputs = model(input_ids=generated_ids, attention_mask=attention_mask, use_cache=True)
    for step in range(5):
        next_token_id = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
        generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

        token_id = next_token_id.item()
        print(f"第 {step + 1} 步：",
            repr(tokenizer.decode([token_id])),
            "| Token ID:", token_id )

        if token_id == tokenizer.eos_token_id:
            print("生成结束。")
            break

        attention_mask = torch.cat([attention_mask, torch.ones_like(next_token_id)], dim=1)

        outputs = model(input_ids=next_token_id, attention_mask=attention_mask, past_key_values=outputs.past_key_values, use_cache=True)
        
print("\n完整文本：")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

print("\n原始输入形状：", tuple(model_inputs["input_ids"].shape))
print("最终序列形状：", tuple(generated_ids.shape))
print("实际新增 Token 数量：", generated_ids.shape[1] - model_inputs["input_ids"].shape[1])

第 1 步： ' I' | Token ID: 358
第 2 步： ' am' | Token ID: 1079
第 3 步： ' studying' | Token ID: 20956
第 4 步： ' for' | Token ID: 369
第 5 步： ' my' | Token ID: 847

完整文本：
Hello, I am a student. I am studying for my

原始输入形状： (1, 7)
最终序列形状： (1, 12)
实际新增 Token 数量： 5


## 9. 推理性能测量：端到端平均生成速度

这一实验为 `model.generate()` 计时，分别新增 8、16、32 个 Token，每种长度重复 3 次。它没有直接给前面的手写循环计时，也没有构造“未复用缓存”与“复用缓存”的性能对照。

### 9.1 预热与设备同步

正式测量前，先用 `max_new_tokens=2, min_new_tokens=2` 调用一次生成，让部分首次调用和运行路径准备发生在计时之外。短序列预热不代表覆盖了全部长序列情况，也不能保证消除所有波动。

`synchronize_device()` 在 CUDA、可用的 NPU/MPS 接口下等待设备任务完成：测量前排除此前排队任务，测量后保证当前任务完成才读耗时。已有输出的设备为 **cpu**，因此本次走的是无需额外同步的分支。支持同步分支不代表代码已经在其他硬件上验证过。

### 9.2 perf_counter、中位数与固定新增长度

`time.perf_counter()` 提供适合测量时间间隔的高精度单调计时。每次调用前记录起点，生成结束并同步后计算差值，收集到 `elapsed_times`。

`max_new_tokens=num_tokens` 限制新增上限，`min_new_tokens=num_tokens` 在达到目标长度前限制 EOS 提前结束；相同取值用于固定本示例的生成工作量。`do_sample=False` 关闭采样，`use_cache=True` 启用生成过程中的缓存。

`statistics.median(elapsed_times)` 取 3 次耗时的中位数，减轻偶发慢调用的影响；这里只得到代表性耗时，不是误差区间或显著性检验。已有输出确认三组实际新增数均达到设定值。

### 9.3 Token/s 测量的范围

`actual_tokens = output_ids.shape[1] - model_inputs["input_ids"].shape[1]` 从完整序列中扣除 Prompt 长度，得到新增 Token 数。速度是：

**Token/s = 实际新增 Token 数 ÷ 三次端到端耗时的中位数**

`output_ids` 来自最后一次重复；当前设置固定新增长度，所以这里将其 Token 数与耗时中位数结合计算。

计时包含整个 `generate()` 调用：Prompt Prefill、后续 Decode 和接口循环开销；模型加载、Tokenizer 编码与解码打印均在计时区间之外。因此它是**端到端平均生成速度**，没有分离首 Token 延迟，也没有排除 Prefill 来测独立的 Decode Throughput。

### 9.4 根据已有输出解读结果

| 实际新增 Token | 中位耗时 | 端到端速度 |
|---:|---:|---:|
| 8 | 1.193 秒 | 6.70 Token/s |
| 16 | 2.168 秒 | 7.38 Token/s |
| 32 | 4.104 秒 | 7.80 Token/s |

这三组保存的 CPU 结果中，生成越长，平均 Token/s 略有增加。Prefill 等固定开销被更多 Token 分摊是可能的解释，但现有计时没有拆分阶段，不能据此确认具体瓶颈或稳态 Decode 速度。

这些数字只代表本次保存的 CPU、输入和运行环境，不推广为 GPU/NPU 性能，也不能用来声称缓存带来了多少倍加速。

In [14]:
import time
import statistics

model.eval()
device = next(model.parameters()).device
model_inputs = inputs.to(device)

def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "npu" and hasattr(torch, "npu"):
        torch.npu.synchronize()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()

with torch.inference_mode():
    _ = model.generate(
        **model_inputs,
        max_new_tokens=2,
        min_new_tokens=2,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

synchronize_device()

generation_lengths = [8, 16, 32]
repeats = 3

print("运行设备：", device)
print("原始输入 Token 数：", model_inputs["input_ids"].shape[1])

for num_tokens in generation_lengths:
    elapsed_times = []

    for _ in range(repeats):
        synchronize_device()
        start_time = time.perf_counter()

        with torch.inference_mode():
            output_ids = model.generate(
                **model_inputs,
                max_new_tokens=num_tokens,
                min_new_tokens=num_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

        synchronize_device()
        elapsed_times.append(time.perf_counter() - start_time)

    elapsed = statistics.median(elapsed_times)
    actual_tokens = (
        output_ids.shape[1] - model_inputs["input_ids"].shape[1]
    )

    print(
        f"生成 {actual_tokens:2d} 个 Token | "
        f"耗时 {elapsed:.3f} 秒 | "
        f"端到端生成速度 {actual_tokens / elapsed:.2f} Token/s"
    )

运行设备： cpu
原始输入 Token 数： 7
生成  8 个 Token | 耗时 1.193 秒 | 端到端生成速度 6.70 Token/s
生成 16 个 Token | 耗时 2.168 秒 | 端到端生成速度 7.38 Token/s
生成 32 个 Token | 耗时 4.104 秒 | 端到端生成速度 7.80 Token/s
